In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import multiprocessing
import os
import pickle
import logging

logging.getLogger("pint").setLevel(logging.ERROR)

if os.environ.get("SLURM_CPUS_PER_TASK"):
    cores = int(os.environ.get("SLURM_CPUS_PER_TASK", 1))
else:
    cores = multiprocessing.cpu_count()
print(f"Number of cores: {cores}")

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(cores)

from functools import partial

import gpjax as gpx
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from astropy.table import Table
from gpjax.kernels import (
    RBF,
    Linear,
    Matern12,
    Periodic,
    Polynomial,
)
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.distributions import Normal
from tqdm import tqdm

from gallifrey.inference import log_likelihood_function
from gallifrey.kernelsearch import (
    KernelSearch,
    get_trainables,
    kernel_summary,
    set_obs_stddev,
)
from gallifrey.mcmc import nuts_warmup, run_mcmc

Number of cores: 8


In [3]:
jax.config.update("jax_enable_x64", True)
rng_key = jax.random.PRNGKey(42)

mode = "save"

## LOAD DATA

In [4]:
model_name = "wasp94Ab_gpmodel"

df = Table.read("../data/external/WASP_94Ab.fit").to_pandas()

# time (adjust for simplicity)
t = df["Time"].to_numpy()
t_min = np.amin(t)
t -= t_min

# white lightcurve
white_lc = df["FluxWL"].to_numpy().T
white_lc_err = df["e_FluxWL"].to_numpy().T

# spectroscopic light curves
y = df.iloc[:, 1:-2:2].to_numpy().T
yerr = df.iloc[:, 2:-1:2].to_numpy().T

# mask out transit
mask = np.ones_like(t, dtype=bool)
mask[100:324] = False

# reference parameter and limb darkening parameter, last entry is white lc
reference = pd.read_csv("../data/external/WASP_94Ab_reference.csv").set_index(
    df.columns[1::2]
)

## KERNEL SEARCH FOR WHITE LC

In [5]:
kernel_library = [
    Linear(),
    RBF(),
    Matern12(),
    Periodic(),
]

In [6]:
if mode == "load":
    with open(
        f"../data/processed/observational_data/gp_models/{model_name}", "rb"
    ) as file:
        model = pickle.load(file)

else:
    tree = KernelSearch(
        kernel_library,
        X=jnp.array(t[mask]),
        y=jnp.array(white_lc[mask]),
        obs_stddev=jnp.amax(white_lc_err[mask]),
        verbosity=1,
    )

    model = tree.search(
        depth=10,
        n_leafs=3,
        patience=1,
    ).posterior

    if mode == "save":
        with open(
            f"../data/processed/observational_data/gp_models/{model_name}", "wb"
        ) as file:
            pickle.dump(model, file)

summary = kernel_summary(model, to_latex=False)

Fitting Layer 1: 100%|██████████| 4/4 [00:06<00:00,  1.60s/it]


Layer 1 || Current top AICs: [-2955.242400488787, -2953.973207503389, -2953.700459425433]


Fitting Layer 2: 100%|██████████| 24/24 [01:10<00:00,  2.94s/it]


Layer 2 || Current top AICs: [-2953.2794747600637, -2952.99855811241, -2951.973207498183]


Fitting Layer 3: 100%|██████████| 32/32 [02:00<00:00,  3.75s/it]


...

Layer 5 || Current top AICs: [-2957.847211318172, -2957.673800155102, -2957.673342517803]


Fitting Layer 6: 100%|██████████| 36/36 [02:25<00:00,  4.04s/it]


Layer 6 || Current top AICs: [-2957.8472113101475, -2957.846608354004, -2955.8472112890813]


Fitting Layer 7: 100%|██████████| 36/36 [02:40<00:00,  4.45s/it]


Layer 7 || Current top AICs: [-2955.847211336998, -2955.847211272088, -2955.8472073421526]
No more improvements found! Terminating early..

Terminated on layer: 7.
Final log likelihood: 1482.923605659086
1482.923605659086
Final number of model parameter: 4
Kernel Summary

Number of Parameter: 4
Kernel Structure: Periodic + Linear • Linear • Linear • Linear
  with obs_stddev = 1.81217e-04 (Trainable : False)

Kernel               Property             Value                Trainable 
--------------------------------------------------------------------------------
Periodic            lengthscale          2.17406e+03          True      

                    variance             1.00140e+00          True      

                    period               3.35245e+03          True      
--------------------------------------------------------------------------------
Linear              variance             7.04685e-03          True      
----------------------------------------------------------

## DEFINE TRANSIT MODEL

In [12]:
def transit_model(
    t, transit_parameter, period=3.9501907, u2=None, global_transit_parameter=None
):
    globals_flag = False if global_transit_parameter is None else True

    if globals_flag:
        scaled_radius = transit_parameter[0]
        limb_darkening_coeff = [transit_parameter[1], u2]
        central_mass = global_transit_parameter[0]
        central_radius = global_transit_parameter[1]
        inclination = jnp.deg2rad(global_transit_parameter[2])
        time_transit = global_transit_parameter[3]
    else:
        scaled_radius = transit_parameter[0]
        limb_darkening_coeff = [transit_parameter[1], u2]
        central_mass = transit_parameter[2]
        central_radius = transit_parameter[3]
        inclination = jnp.deg2rad(transit_parameter[4])
        time_transit = transit_parameter[5]

    central = orbits.keplerian.Central(
        mass=central_mass,
        radius=central_radius,
    )

    orbit = orbits.keplerian.Body(
        central=central,
        period=period,
        radius=scaled_radius * central.radius,
        inclination=inclination,
        time_transit=time_transit,
    )

    lc = LimbDarkLightCurve(limb_darkening_coeff).light_curve(orbit, t=t)
    return lc

## FIT TRANSIT PARAMETER FOR WHITE LC

In [13]:
white_lc_log_likelihood = log_likelihood_function(
    model,
    jit(
        partial(transit_model, u2=reference["u2"]["FluxWL"])
    ),  # partial returns new function with u2 fixed
    t,
    white_lc,
    mask,
    fix_gp=False,
    compile=True,
    negative=True,
)

x0 = {
    "gp_parameter": get_trainables(model, unconstrain=True),
    # planet radius, u1, host_star_mass, host_star_radius, inclination, time_transit
    "lc_parameter": jnp.array([0.11, 0.53, 1.45, 1.653, 89.3, 0.18]),
}
white_lc_solve = ScipyMinimize(
    fun=white_lc_log_likelihood,
    method="l-bfgs-b",
).run(x0)
white_lc_parameter = white_lc_solve.params["lc_parameter"]

## DEFINE LIKELIHOOD, PRIOR, POSTERIOR

In [15]:
def get_logprob(gp_model, y, yerr, u1, u2, initial_position=None, fix_gp=False):
    if initial_position is None:
        initial_position = {
            "gp_parameter": get_trainables(gp_model, unconstrain=True),
            "lc_parameter": jnp.array([0.10, u1]),
        }

    param_priors = {
        "gp_parameter": Normal(
            loc=initial_position["gp_parameter"],
            scale=0.1 * jnp.abs(initial_position["gp_parameter"]),
        ),
        "lc_parameter": Normal(
            loc=initial_position["lc_parameter"],
            scale=[0.2, 0.05],
        ),
    }

    # define transit light curve model
    lc_model = jit(
        partial(
            transit_model,
            u2=u2,
            global_transit_parameter=white_lc_parameter[2:],
        )
    )

    # update observational uncertainty data uncertainty
    gp_model = set_obs_stddev(gp_model, jnp.amax(yerr[mask]))

    log_likelihood = log_likelihood_function(
        gp_model,
        lc_model,
        t,
        y,
        mask,
        fix_gp=fix_gp,
        compile=True,
    )

    @jit
    def log_priors(params):
        gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
        lc_log_priors = param_priors["lc_parameter"].log_prob(params["lc_parameter"])
        return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)

    @jit
    def log_probability(params):
        return log_likelihood(params) + log_priors(params)

    return log_probability, initial_position

## PERFORM FITS

In [16]:
parameter_solutions = []
for i in tqdm(range(len(y))):
    log_probability, initial_position = get_logprob(
        model,
        y[i],
        yerr[i],
        reference["u1_linear"].iloc[i],
        reference["u2"].iloc[i],
    )

    solve = ScipyMinimize(
        fun=jit(lambda par: -log_probability(par)), method="l-bfgs-b"
    ).run(initial_position)
    parameter_solutions.append(solve.params)

100%|██████████| 19/19 [02:20<00:00,  7.41s/it]


In [19]:
# import matplotlib.pyplot as plt

# plt.scatter(range(len(y)), [sol["lc_parameter"][0] for sol in parameter_solutions])
# plt.scatter(range(len(y)), reference["Rp_linear"][:-1])
# plt.scatter(range(len(y)), reference["Rp_gp"][:-1])

# plt.figure()
# plt.scatter(
#     range(len(y)),
#     (
#         reference["Rp_linear"][:-1]
#         - [sol["lc_parameter"][0] for sol in parameter_solutions]
#     )
#     / reference["Rp_linear"][:-1],
# )
# plt.scatter(
#     range(len(y)),
#     (reference["Rp_gp"][:-1] - [sol["lc_parameter"][0] for sol in parameter_solutions])
#     / reference["Rp_gp"][:-1],
# )

## RUN MCMC

In [ ]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 150
num_samples = 80
num_chains = cores

fix_gp = False

In [ ]:
rng_key, warmup_key = jax.random.split(rng_key, 2)

# run nuts adaption on white lc
log_probability, initial_position = get_logprob(
    model,
    white_lc,
    white_lc_err,
    reference["u1_linear"]["FluxWL"],
    reference["u2"]["FluxWL"],
    fix_gp=fix_gp,
)

state, parameters = nuts_warmup(
    warmup_key,
    log_probability,
    initial_position,
    num_steps=num_adapt,
)

In [ ]:
chains = {"gp_parameter": [], "lc_parameter": []}

if mode == "load":
    chains = np.load(
        f"../data/processed/observational_data/mcmc_chains/{model_name}_parameter.npz"
    )

else:
    for i in tqdm(range(len(y))):
        log_probability, initial_position = get_logprob(
            model,
            y[i],
            yerr[i],
            reference["u1_linear"].iloc[i],
            reference["u2"].iloc[i],
            initial_position=parameter_solutions[i],
            fix_gp=fix_gp,
        )

        # define initial positions and add scatter
        initial_positions = {}
        for key, value in initial_position.items():
            rng_key, scatter_key = jax.random.split(rng_key)
            initial_positions[key] = jnp.tile(
                value, (num_chains, 1)
            ) + 0.05 * value * jax.random.normal(
                scatter_key, shape=(num_chains, value.size)
            )

        rng_key, sample_key = jax.random.split(rng_key, 2)

        final_state, state_history, info_history = run_mcmc(
            sample_key,
            log_probability,
            parameters,
            initial_positions,
            num_steps=num_samples,
        )

        for par in ["gp_parameter", "lc_parameter"]:
            chain = np.array(state_history.position[par])
            chains[par].append(chain)

    if mode == "save":
        np.savez(
            f"../data/processed/observational_data/mcmc_chains/{model_name}_parameter.npz",
            **chains,
        )